What Is a Gateway?
A gateway is a proxy layer between your application and any LLM API:

Your App  →  [LLM Gateway]  →  Groq / NVIDIA / OpenAI / Anthropic
The gateway intercepts every request and can:

Route to the right provider
Retry automatically on transient failures
Fallback to a different provider if the primary fails
Balance load across multiple models by weight
Cache responses so identical requests never hit the LLM twice
Log everything with full request/response detail
Tag requests with metadata (user, session, feature) for analytics
Enforce timeouts and kill slow requests

Why Portkey?
Feature	           Portkey	LiteLLM	Direct SDK
250+ models unified	✅	✅	❌ one provider
Automatic fallbacks	✅	✅	❌ manual
Request caching	✅	✅	❌ manual
Observability dashboard	✅ beautiful UI	⚠️ basic	❌ none
LangChain drop-in	✅	✅	✅ native
Config-as-code	✅ JSON/YAML	❌	❌
Open source	✅ Apache 2.0	✅ MIT	N/A
Overhead	~20–40ms	~30ms	0ms

In [ ]:
import os
import time
import uuid
import json
from dotenv import load_dotenv
from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL

load_dotenv(dotenv_path="../.env")  # Load environment variables from .env file
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")
if not PORTKEY_API_KEY:
    raise ValueError("PORTKEY_API_KEY is missing. Set it in ../.env before running this notebook.")
PORTKEY_CONFIG_ID = "flight-policy"

GROQ_SLUG    =  "flight-policy"  # your Groq integration slug
GROQ_MODEL   = f"@{GROQ_SLUG}/llama-3.3-70b-versatile"

# Second Groq integration for multi-provider experiments (5, 6, 10)
# Uses a smaller/faster model as the fallback target
GROQ_SLUG_2      =  "flight-policy"           # your second Groq slug
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/llama-3.1-8b-instant"

# Keep GROQ_API_KEY for the LangChain experiment (Exp 9)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [9]:
def section(title):
    print(f"\n{'='*62}")
    print(f"  {title}")
    print(f"{'='*62}")

def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱  {ms:.0f}ms{note}")
    print(bar)

# Main Portkey client — used for all simple experiments
portkey = Portkey(api_key=PORTKEY_API_KEY)

print("Setup complete!")
print(f"  Portkey API Key : {'OK' if PORTKEY_API_KEY else 'MISSING'}")
print(f"  Groq slug       : {GROQ_SLUG}")
print(f"  Groq model ref  : {GROQ_MODEL}")
print(f"  Groq slug 2     : {GROQ_SLUG_2}")
print(f"  Small model ref : {GROQ_MODEL_SMALL}")
print(f"\nPortkey Gateway : {PORTKEY_GATEWAY_URL}")

Setup complete!
  Portkey API Key : OK
  Groq slug       : flight-policsy
  Groq model ref  : @flight-policsy/llama-3.3-70b-versatile
  Groq slug 2     : flight-policy
  Small model ref : @flight-policy/llama-3.1-8b-instant

Portkey Gateway : https://api.portkey.ai/v1


In [10]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
raw_groq = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.3-70b-versatile", temperature=0)
section("BASELINE — Direct Groq Call")
questions = [
    "What is Kubernetes in one sentence?",
    "What is Intel SRIOV?",
]
for q in questions:
    start = time.time()
    r = raw_groq.invoke([HumanMessage(content=q)])
    show(q, r.content, (time.time() - start) * 1000, label="Direct Groq Call")


  BASELINE — Direct Groq Call

──────────────────────────────────────────────────────────────
Q: What is Kubernetes in one sentence?
A: Kubernetes is an open-source container orchestration system that automates the deployment, scaling, and management of containerized applications across a cluster of machines.
⏱  311ms | Direct Groq Call
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is Intel SRIOV?
A: Intel SR-IOV (Single Root I/O Virtualization) is a technology that allows a single physical device, such as a network interface card (NIC) or a storage controller, to appear as multiple virtual devices to the operating system and applications. This is achieved...
⏱  1727ms | Direct Groq Call
──────────────────────────────────────────────────────────────


In [11]:
section("EXP 1 — Basic Gateway Call")
for q in questions:
    start = time.time()
    response = portkey.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": q}],)
    show(q, response.choices[0].message.content, (time.time() - start) * 1000, label="Basic Gateway Call")
print("\n✅ Check portkey.ai → Logs to see both requests fully logged!")
print("   Token count, cost, latency — all tracked. Zero extra code.")


  EXP 1 — Basic Gateway Call


BadRequestError: Error code: 400 - {'status': 'failure', 'message': 'Following keys are not valid: flight-policsy'}